In [ ]:
# Run this cell first to install required packages in JupyterLite
%pip install -q numpy matplotlib ipywidgets scikit-learn


# Figure 17.15: Learnable upsampling using transposed convolution


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

def transposed_conv(x, kernel, stride=2, padding=0):
    """
    Compute transposed convolution output.
    x: (H_in, W_in), kernel: (k, k)
    Returns output (H_out, W_out)
    """
    H_in, W_in = x.shape
    k = kernel.shape[0]
    H_out = (H_in - 1) * stride - 2 * padding + k
    W_out = (W_in - 1) * stride - 2 * padding + k
    out = np.zeros((H_out, W_out))
    for i in range(H_in):
        for j in range(W_in):
            r_start = i * stride - padding
            c_start = j * stride - padding
            for ki in range(k):
                for kj in range(k):
                    r, c = r_start + ki, c_start + kj
                    if 0 <= r < H_out and 0 <= c < W_out:
                        out[r, c] += x[i, j] * kernel[ki, kj]
    return out

def draw_transposed_conv(H_in=3, kernel_size=2, stride=2, padding=0, show_step=0):
    k = kernel_size
    H_out = (H_in - 1) * stride - 2 * padding + k

    x = np.array([[i * H_in + j + 1 for j in range(H_in)] for i in range(H_in)], dtype=float)
    kernel = np.ones((k, k)) if k == 2 else np.array([[1,0,1],[0,1,0],[1,0,1]], dtype=float)

    full_out = transposed_conv(x, kernel, stride, padding)

    # Partial output up to show_step input elements
    partial_out = np.zeros_like(full_out)
    positions = [(i, j) for i in range(H_in) for j in range(H_in)]
    for step in range(min(show_step, len(positions))):
        i, j = positions[step]
        r_start = i * stride - padding
        c_start = j * stride - padding
        for ki in range(k):
            for kj in range(k):
                r, c = r_start + ki, c_start + kj
                if 0 <= r < full_out.shape[0] and 0 <= c < full_out.shape[1]:
                    partial_out[r, c] += x[i, j] * kernel[ki, kj]

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    def show_grid(ax, data, title, highlight=None, cmap='Blues'):
        vmax = max(data.max(), 1)
        ax.imshow(data, cmap=cmap, vmin=0, vmax=vmax, aspect='equal')
        for i in range(data.shape[0]):
            for j in range(data.shape[1]):
                v = data[i, j]
                ax.text(j, i, f'{v:.0f}', ha='center', va='center',
                        fontsize=12, fontweight='bold',
                        color='white' if v > vmax * 0.6 else 'black')
        if highlight:
            from matplotlib.patches import Rectangle
            r, c = highlight
            ax.add_patch(Rectangle((c-0.5, r-0.5), 1, 1, fill=False, edgecolor='#dc2626', lw=3))
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.set_xticks([]); ax.set_yticks([])

    cur = positions[show_step-1] if show_step > 0 else None
    show_grid(axes[0], x, f'Input ({H_in}x{H_in})', highlight=cur)
    show_grid(axes[1], kernel, f'Kernel ({k}x{k})', cmap='Greens')

    out_to_show = partial_out if show_step < len(positions) else full_out
    show_grid(axes[2], out_to_show,
              f'Output ({H_out}x{H_out})\nstep {show_step}/{len(positions)}', cmap='Purples')

    formula = (f'H_out = (H_in-1)*s - 2p + k = '
               f'({H_in}-1)*{stride} - 2*{padding} + {k} = {H_out}')
    if cur:
        r_s = cur[0] * stride - padding
        c_s = cur[1] * stride - padding
        formula += f'\nActive cell ({cur[0]},{cur[1]}): kernel placed at output ({r_s},{c_s})'
    plt.suptitle(formula, fontsize=9, y=0.0)
    plt.tight_layout()
    plt.show()

n_steps = 36  # max steps for 6x6 input
widgets.interact(draw_transposed_conv,
    H_in=widgets.IntSlider(min=2, max=5, step=1, value=3, description='Input size', continuous_update=False),
    kernel_size=widgets.Dropdown(options=[('2x2', 2), ('3x3', 3)], value=2, description='Kernel size'),
    stride=widgets.IntSlider(min=1, max=3, step=1, value=2, description='Stride s', continuous_update=False),
    padding=widgets.IntSlider(min=0, max=2, step=1, value=0, description='Padding p', continuous_update=False),
    show_step=widgets.IntSlider(min=0, max=n_steps, step=1, value=0, description='Step', continuous_update=False),
)